# Chapter 9 Lab — Orchestration

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 9 — Orchestration  
**Goal:** Move beyond ad-hoc chaining — learn state machines, graph-based workflows, and production orchestration patterns with LangGraph.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Ad-Hoc Chaining Problems |
| 3 | State Machines for Agents |
| 4 | Graph-Based Workflows |
| 5 | LangGraph Concepts — nodes, edges, state, conditional routing |
| 6 | Routing Patterns |
| 7 | Interrupts and Human-in-the-Loop |
| 8 | Checkpointing |
| 9 | Error Handling in Graphs |
| 10 | Exercises |

> **Note:** This lab uses **mock LLM responses** so every cell runs without an API key. Replace the mock helpers with real LLM calls when you are ready to go live.

---
## 1. Setup

In [ ]:
%pip install langgraph langchain-openai langchain-core --quiet

In [ ]:
import json
import operator
import random
import textwrap
import time
from enum import Enum
from typing import Annotated, Any, Literal, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print("All imports succeeded.")

In [ ]:
# ── Mock LLM helper ───────────────────────────────────────────────────────────
# Returns canned responses so the notebook runs without an API key.

def mock_llm(prompt: str, **kwargs) -> str:
    """Simulate an LLM call. Returns a deterministic response based on keywords."""
    p = prompt.lower()
    if "classify" in p or "route" in p or "category" in p:
        if "refund" in p or "billing" in p:
            return "billing"
        elif "bug" in p or "error" in p or "crash" in p:
            return "technical"
        elif "feature" in p or "request" in p:
            return "feature_request"
        return "general"
    if "summarize" in p or "summary" in p:
        return "This is a mock summary of the provided content."
    if "draft" in p or "write" in p or "respond" in p:
        return "Thank you for reaching out. We are looking into your request and will follow up shortly."
    if "review" in p or "check" in p or "approve" in p:
        return "APPROVED"
    if "extract" in p:
        return json.dumps({"name": "Jane Doe", "issue": "billing discrepancy", "priority": "high"})
    return f"Mock response for: {prompt[:60]}..."


print("mock_llm ready:", mock_llm("Classify this ticket: I want a refund"))

---
## 2. Ad-Hoc Chaining Problems

The simplest orchestration approach is **sequential chaining**: pipe the output of one LLM call into the next. This works for toy demos but fails in production because:

1. **No branching** — every input follows the same path regardless of content.  
2. **No state** — if step 3 fails, you must re-run steps 1 and 2.  
3. **No visibility** — you cannot inspect or replay intermediate steps.  
4. **No human oversight** — there is no place to insert approval gates.

In [ ]:
# ── Naive sequential chain ────────────────────────────────────────────────────

def naive_chain(ticket: str) -> dict:
    """Process a support ticket with a simple sequential chain."""
    # Step 1 — classify
    category = mock_llm(f"Classify this ticket: {ticket}")
    # Step 2 — draft response
    draft = mock_llm(f"Draft a response for this {category} ticket: {ticket}")
    # Step 3 — review
    review = mock_llm(f"Review this draft: {draft}")
    return {"category": category, "draft": draft, "review": review}


result = naive_chain("I was charged twice for my subscription")
for k, v in result.items():
    print(f"{k:>10}: {v}")

In [ ]:
# ── Problem demo: a failure in step 2 forces a full restart ───────────────────

call_count = 0

def flaky_llm(prompt: str) -> str:
    """Simulates a call that fails intermittently."""
    global call_count
    call_count += 1
    if call_count == 2:  # fail on the second call
        raise ConnectionError("Simulated API timeout")
    return mock_llm(prompt)


def naive_chain_flaky(ticket: str) -> dict:
    category = flaky_llm(f"Classify this ticket: {ticket}")  # call 1 — OK
    draft = flaky_llm(f"Draft a response for {category}: {ticket}")  # call 2 — FAILS
    review = flaky_llm(f"Review: {draft}")
    return {"category": category, "draft": draft, "review": review}


try:
    call_count = 0
    naive_chain_flaky("My app crashes on login")
except ConnectionError as e:
    print(f"Chain failed at step 2: {e}")
    print("Step 1 result is lost — the entire chain must restart.")
    print("This is the core problem with ad-hoc chaining.")

**Takeaway:** Ad-hoc chains are fragile, opaque, and cannot branch. We need a structured execution model.

---
## 3. State Machines for Agents

A **state machine** gives us:
- Explicit states (`classifying`, `drafting`, `reviewing`, `done`)
- Defined transitions between states
- The ability to resume from a known state after failure

Below we implement a minimal state machine from scratch — no frameworks — to show the concept.

In [ ]:
# ── Minimal state machine (no framework) ──────────────────────────────────────

class TicketState(Enum):
    CLASSIFY = "classify"
    DRAFT = "draft"
    REVIEW = "review"
    DONE = "done"


class TicketStateMachine:
    """Process a support ticket through explicit states."""

    def __init__(self, ticket: str):
        self.ticket = ticket
        self.state = TicketState.CLASSIFY
        self.data: dict[str, Any] = {}
        self.history: list[str] = []

    def step(self) -> bool:
        """Execute one transition. Returns True if there is more work."""
        self.history.append(f"Entering {self.state.value}")

        if self.state == TicketState.CLASSIFY:
            self.data["category"] = mock_llm(f"Classify this ticket: {self.ticket}")
            self.state = TicketState.DRAFT
            return True

        elif self.state == TicketState.DRAFT:
            cat = self.data["category"]
            self.data["draft"] = mock_llm(f"Draft a response for {cat}: {self.ticket}")
            self.state = TicketState.REVIEW
            return True

        elif self.state == TicketState.REVIEW:
            self.data["review"] = mock_llm(f"Review this draft: {self.data['draft']}")
            self.state = TicketState.DONE
            return True

        else:  # DONE
            return False

    def run(self) -> dict:
        while self.step():
            pass
        return self.data


sm = TicketStateMachine("I was charged twice")
result = sm.run()
print("History:", sm.history)
print("Result: ", result)

In [ ]:
# ── Resume after failure demo ─────────────────────────────────────────────────

sm2 = TicketStateMachine("App crashes on login")
sm2.step()  # classify — succeeds
print(f"After step 1 -> state={sm2.state.value}, data={sm2.data}")

# Simulate: save state, "crash", then restore and continue
saved_state = sm2.state
saved_data = sm2.data.copy()
print(f"Saved checkpoint: state={saved_state.value}")

# "Restore" into a new machine
sm3 = TicketStateMachine("App crashes on login")
sm3.state = saved_state
sm3.data = saved_data
result = sm3.run()
print(f"Resumed and finished: {result}")
print(f"History (resumed): {sm3.history}")

**Key insight:** The state machine can be interrupted, serialized, and resumed. This is the foundation that graph-based orchestration frameworks build on.

---
## 4. Graph-Based Workflows

A **directed graph** generalizes the state machine:
- **Nodes** = processing steps (functions)
- **Edges** = transitions between steps
- **Conditional edges** = routing logic ("if billing, go to node A; if technical, go to node B")

Below we build a simple graph from scratch before switching to LangGraph.

In [ ]:
# ── DIY directed graph executor ───────────────────────────────────────────────

class SimpleGraph:
    """Minimal directed graph executor."""

    def __init__(self):
        self.nodes: dict[str, callable] = {}
        self.edges: dict[str, str | callable] = {}  # node -> next_node or routing fn
        self.entry: str | None = None

    def add_node(self, name: str, fn: callable):
        self.nodes[name] = fn
        if self.entry is None:
            self.entry = name

    def add_edge(self, src: str, dst: str):
        self.edges[src] = dst

    def add_conditional_edge(self, src: str, router: callable):
        """router(state) -> next node name."""
        self.edges[src] = router

    def run(self, state: dict) -> dict:
        current = self.entry
        trace = []
        while current and current != "__end__":
            trace.append(current)
            fn = self.nodes[current]
            state = fn(state)
            nxt = self.edges.get(current)
            if callable(nxt):
                current = nxt(state)
            else:
                current = nxt
        state["_trace"] = trace
        return state


# ── Build a ticket-processing graph ────────────────────────────────────────────

def classify_node(state):
    state["category"] = mock_llm(f"Classify: {state['ticket']}")
    return state

def billing_node(state):
    state["response"] = "Billing team will review your charge within 24 hours."
    return state

def technical_node(state):
    state["response"] = "A support engineer has been assigned to your issue."
    return state

def general_node(state):
    state["response"] = mock_llm(f"Draft a general response for: {state['ticket']}")
    return state

def route_by_category(state):
    return {"billing": "billing", "technical": "technical"}.get(state["category"], "general")


g = SimpleGraph()
g.add_node("classify", classify_node)
g.add_node("billing", billing_node)
g.add_node("technical", technical_node)
g.add_node("general", general_node)

g.add_conditional_edge("classify", route_by_category)
g.add_edge("billing", "__end__")
g.add_edge("technical", "__end__")
g.add_edge("general", "__end__")

# Test with different tickets
for ticket in ["I want a refund", "App crashes on login", "Can you add dark mode?"]:
    out = g.run({"ticket": ticket})
    print(f"Ticket: {ticket!r}")
    print(f"  Route: {' -> '.join(out['_trace'])}")
    print(f"  Category: {out['category']}")
    print(f"  Response: {out['response']}\n")

Our DIY graph works but lacks checkpointing, interrupts, and error handling. That is exactly what **LangGraph** provides.

---
## 5. LangGraph Concepts

LangGraph models agent workflows as **stateful graphs**. The four core concepts:

| Concept | Description |
|---------|-------------|
| **State** | A `TypedDict` that flows through the graph. Each node reads and writes to it. |
| **Node** | A Python function `(state) -> partial_state_update`. |
| **Edge** | A fixed connection from one node to another. |
| **Conditional Edge** | A routing function `(state) -> node_name` that picks the next node at runtime. |

Let us rebuild our ticket processor in LangGraph.

In [ ]:
# ── LangGraph state definition ────────────────────────────────────────────────

class TicketGraphState(TypedDict):
    ticket: str
    category: str
    response: str
    review: str
    messages: Annotated[list, add_messages]  # append-only message log


print("State schema fields:", list(TicketGraphState.__annotations__.keys()))

In [ ]:
# ── Node functions ────────────────────────────────────────────────────────────
# Each node receives the full state and returns a PARTIAL update dict.

def classify(state: TicketGraphState) -> dict:
    category = mock_llm(f"Classify this support ticket into a category: {state['ticket']}")
    return {"category": category}


def draft_response(state: TicketGraphState) -> dict:
    draft = mock_llm(
        f"Draft a customer response for this {state['category']} ticket: {state['ticket']}"
    )
    return {"response": draft}


def review_response(state: TicketGraphState) -> dict:
    verdict = mock_llm(f"Review and approve this draft response: {state['response']}")
    return {"review": verdict}


print("Node functions defined: classify, draft_response, review_response")

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────────────

workflow = StateGraph(TicketGraphState)

# Add nodes
workflow.add_node("classify", classify)
workflow.add_node("draft_response", draft_response)
workflow.add_node("review_response", review_response)

# Add edges (linear for now)
workflow.add_edge(START, "classify")
workflow.add_edge("classify", "draft_response")
workflow.add_edge("draft_response", "review_response")
workflow.add_edge("review_response", END)

# Compile
app = workflow.compile()
print("Graph compiled successfully.")
print("Nodes:", list(app.get_graph().nodes.keys()))

In [ ]:
# ── Run the graph ─────────────────────────────────────────────────────────────

initial_state = {
    "ticket": "I was charged twice for my subscription last month",
    "category": "",
    "response": "",
    "review": "",
    "messages": [],
}

final = app.invoke(initial_state)

print(f"Category: {final['category']}")
print(f"Response: {final['response']}")
print(f"Review:   {final['review']}")

In [ ]:
# ── Stream mode: watch each node execute ──────────────────────────────────────

print("Streaming graph execution:\n")
for event in app.stream(initial_state):
    for node_name, update in event.items():
        print(f"  [{node_name}] returned: {update}")

---
## 6. Routing Patterns

A **router** uses a conditional edge to send the state down different paths based on content. This is one of the most common orchestration patterns:

```
START -> classify --billing--> billing_handler -> review -> END
                  --technical-> tech_handler ----> review -> END
                  --general---> general_handler -> review -> END
```

In [ ]:
# ── State with routing ────────────────────────────────────────────────────────

class RouterState(TypedDict):
    ticket: str
    category: str
    response: str
    review: str


# ── Node functions ────────────────────────────────────────────────────────────

def classify_ticket(state: RouterState) -> dict:
    cat = mock_llm(f"Classify this ticket into a route category: {state['ticket']}")
    print(f"  [classify] -> {cat}")
    return {"category": cat}


def handle_billing(state: RouterState) -> dict:
    print("  [billing handler]")
    return {"response": "Billing: We will process your refund within 3-5 business days."}


def handle_technical(state: RouterState) -> dict:
    print("  [technical handler]")
    return {"response": "Technical: A support engineer will contact you within 2 hours."}


def handle_general(state: RouterState) -> dict:
    print("  [general handler]")
    return {"response": mock_llm(f"Respond to: {state['ticket']}")}


def review_node(state: RouterState) -> dict:
    verdict = mock_llm(f"Approve this response: {state['response']}")
    print(f"  [review] -> {verdict}")
    return {"review": verdict}


# ── Router function ───────────────────────────────────────────────────────────

def route_ticket(state: RouterState) -> str:
    """Conditional edge: pick the next node based on category."""
    mapping = {
        "billing": "handle_billing",
        "technical": "handle_technical",
    }
    return mapping.get(state["category"], "handle_general")


print("Router components defined.")

In [ ]:
# ── Build the routing graph ───────────────────────────────────────────────────

router_wf = StateGraph(RouterState)

router_wf.add_node("classify_ticket", classify_ticket)
router_wf.add_node("handle_billing", handle_billing)
router_wf.add_node("handle_technical", handle_technical)
router_wf.add_node("handle_general", handle_general)
router_wf.add_node("review", review_node)

router_wf.add_edge(START, "classify_ticket")
router_wf.add_conditional_edges(
    "classify_ticket",
    route_ticket,
    {"handle_billing": "handle_billing", "handle_technical": "handle_technical", "handle_general": "handle_general"},
)
router_wf.add_edge("handle_billing", "review")
router_wf.add_edge("handle_technical", "review")
router_wf.add_edge("handle_general", "review")
router_wf.add_edge("review", END)

router_app = router_wf.compile()
print("Router graph compiled.")

In [ ]:
# ── Test routing with different ticket types ──────────────────────────────────

test_tickets = [
    "I was billed twice for my subscription",
    "The app crashes when I click the settings button",
    "Can you add a feature to export reports as PDF?",
]

for ticket in test_tickets:
    print(f"\nTicket: {ticket!r}")
    result = router_app.invoke({"ticket": ticket, "category": "", "response": "", "review": ""})
    print(f"  Final response: {result['response']}")
    print(f"  Review verdict: {result['review']}")

---
## 7. Interrupts and Human-in-the-Loop (HITL)

Production systems often need a **human approval gate**: the graph pauses, a human reviews, and the graph resumes. LangGraph supports this via `interrupt_before` or `interrupt_after` on specific nodes.

When the graph hits an interrupt, it stops and returns partial state. You can then inspect, modify state, and resume.

In [ ]:
# ── Build a graph with a human interrupt before the review step ────────────────

class HITLState(TypedDict):
    ticket: str
    category: str
    draft: str
    human_approved: bool
    final_response: str


def hitl_classify(state: HITLState) -> dict:
    return {"category": mock_llm(f"Classify: {state['ticket']}")}


def hitl_draft(state: HITLState) -> dict:
    return {"draft": mock_llm(f"Draft response for {state['category']}: {state['ticket']}")}


def hitl_finalize(state: HITLState) -> dict:
    """This node runs AFTER the human approves."""
    if state.get("human_approved"):
        return {"final_response": f"SENT: {state['draft']}"}
    else:
        return {"final_response": "BLOCKED: Human did not approve the draft."}


hitl_wf = StateGraph(HITLState)
hitl_wf.add_node("classify", hitl_classify)
hitl_wf.add_node("draft", hitl_draft)
hitl_wf.add_node("finalize", hitl_finalize)

hitl_wf.add_edge(START, "classify")
hitl_wf.add_edge("classify", "draft")
hitl_wf.add_edge("draft", "finalize")
hitl_wf.add_edge("finalize", END)

# Compile with interrupt BEFORE finalize — the graph pauses here
memory = MemorySaver()
hitl_app = hitl_wf.compile(checkpointer=memory, interrupt_before=["finalize"])

print("HITL graph compiled with interrupt before 'finalize'.")

In [ ]:
# ── Run until interrupt ───────────────────────────────────────────────────────

config = {"configurable": {"thread_id": "hitl-demo-1"}}

initial = {
    "ticket": "I need a refund for order #12345",
    "category": "",
    "draft": "",
    "human_approved": False,
    "final_response": "",
}

# This will run classify -> draft, then PAUSE before finalize
partial_state = hitl_app.invoke(initial, config=config)

print("Graph paused before 'finalize'.")
print(f"  Category: {partial_state['category']}")
print(f"  Draft:    {partial_state['draft']}")
print(f"  Approved: {partial_state['human_approved']}")
print("\n  --> A human reviewer would inspect the draft here.")

In [ ]:
# ── Human approves and resumes ────────────────────────────────────────────────

# Simulate human approval by updating state
hitl_app.update_state(config, {"human_approved": True})

# Resume execution — finalize will now run
final = hitl_app.invoke(None, config=config)

print("Graph resumed after human approval.")
print(f"  Final response: {final['final_response']}")

In [ ]:
# ── Demo: human REJECTS the draft ─────────────────────────────────────────────

config2 = {"configurable": {"thread_id": "hitl-demo-2"}}

# Run to interrupt
partial = hitl_app.invoke(
    {"ticket": "Charge me for wrong item", "category": "", "draft": "",
     "human_approved": False, "final_response": ""},
    config=config2,
)
print(f"Draft: {partial['draft']}")

# Human rejects — do NOT set human_approved to True
final2 = hitl_app.invoke(None, config=config2)
print(f"Final: {final2['final_response']}")
print("The draft was blocked because the human did not approve.")

---
## 8. Checkpointing

Checkpointing saves the graph state after each node so you can:
1. **Resume** after a crash without re-running completed steps
2. **Replay** a specific execution for debugging
3. **Fork** a conversation at any point ("what if" analysis)

LangGraph's `MemorySaver` stores checkpoints in memory. In production you would use `SqliteSaver` or `PostgresSaver`.

In [ ]:
# ── Build a checkpointed graph ────────────────────────────────────────────────

class PipelineState(TypedDict):
    document: str
    extracted: str
    summary: str
    status: str


def extract_info(state: PipelineState) -> dict:
    print("  [extract_info] running...")
    extracted = mock_llm(f"Extract key information from: {state['document']}")
    return {"extracted": extracted, "status": "extracted"}


def summarize_doc(state: PipelineState) -> dict:
    print("  [summarize] running...")
    summary = mock_llm(f"Summarize: {state['extracted']}")
    return {"summary": summary, "status": "summarized"}


def finalize_doc(state: PipelineState) -> dict:
    print("  [finalize] running...")
    return {"status": "complete"}


cp_wf = StateGraph(PipelineState)
cp_wf.add_node("extract", extract_info)
cp_wf.add_node("summarize", summarize_doc)
cp_wf.add_node("finalize", finalize_doc)
cp_wf.add_edge(START, "extract")
cp_wf.add_edge("extract", "summarize")
cp_wf.add_edge("summarize", "finalize")
cp_wf.add_edge("finalize", END)

cp_memory = MemorySaver()
cp_app = cp_wf.compile(checkpointer=cp_memory)
print("Checkpointed pipeline ready.")

In [ ]:
# ── Run and inspect checkpoints ───────────────────────────────────────────────

cp_config = {"configurable": {"thread_id": "doc-pipeline-1"}}

result = cp_app.invoke(
    {"document": "Patient Jane Doe presented with symptoms of...",
     "extracted": "", "summary": "", "status": "new"},
    config=cp_config,
)

print(f"\nFinal status: {result['status']}")
print(f"Summary: {result['summary']}")

In [ ]:
# ── List all checkpoints for this thread ──────────────────────────────────────

print("Checkpoint history for thread 'doc-pipeline-1':\n")
for i, cp in enumerate(cp_memory.list(cp_config)):
    meta = cp.metadata
    state_snapshot = cp.checkpoint
    step = meta.get("step", "?")
    source = meta.get("source", "?")
    print(f"  Checkpoint {i}: step={step}, source={source}, id={cp.config['configurable']['checkpoint_id'][:16]}...")

In [ ]:
# ── Replay from a specific checkpoint (get_state) ─────────────────────────────

snapshot = cp_app.get_state(cp_config)
print("Current state from latest checkpoint:")
print(f"  Status:    {snapshot.values.get('status')}")
print(f"  Summary:   {snapshot.values.get('summary')}")
print(f"  Next node: {snapshot.next}")
print("\nCheckpointing lets you resume, fork, or replay any execution.")

---
## 9. Error Handling in Graphs

Production graphs need resilience:
- **Retry** — re-execute a node a fixed number of times
- **Fallback** — if a node keeps failing, route to a fallback path
- **Dead-letter** — record the failure and move on

LangGraph does not have built-in retry decorators, so we implement them at the node level.

In [ ]:
# ── Retry decorator ───────────────────────────────────────────────────────────

import functools

def retry_node(max_retries: int = 3, delay: float = 0.1):
    """Decorator that retries a LangGraph node function on failure."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(state):
            last_error = None
            for attempt in range(1, max_retries + 1):
                try:
                    return func(state)
                except Exception as e:
                    last_error = e
                    print(f"    [retry] {func.__name__} attempt {attempt}/{max_retries} failed: {e}")
                    time.sleep(delay)
            # All retries exhausted — return error in state
            return {"error": f"{func.__name__} failed after {max_retries} retries: {last_error}"}
        return wrapper
    return decorator


print("retry_node decorator defined.")

In [ ]:
# ── Graph with retry and fallback ─────────────────────────────────────────────

class RobustState(TypedDict):
    input_text: str
    result: str
    error: str
    attempts: int


flaky_counter = {"n": 0}


@retry_node(max_retries=3, delay=0.05)
def flaky_process(state: RobustState) -> dict:
    """Simulates a node that fails the first 2 times, then succeeds."""
    flaky_counter["n"] += 1
    if flaky_counter["n"] <= 2:
        raise ConnectionError(f"Simulated failure #{flaky_counter['n']}")
    return {"result": f"Processed: {state['input_text']}", "attempts": flaky_counter["n"]}


def fallback_process(state: RobustState) -> dict:
    """Fallback: runs if flaky_process reported an error."""
    print("  [fallback] using cached/default response")
    return {"result": "Fallback response: please try again later.", "error": ""}


def check_error(state: RobustState) -> str:
    """Route to fallback if there was an error, otherwise to END."""
    if state.get("error"):
        return "fallback"
    return "done"


print("Robust nodes defined.")

In [ ]:
# ── Build and run the robust graph ────────────────────────────────────────────

robust_wf = StateGraph(RobustState)
robust_wf.add_node("process", flaky_process)
robust_wf.add_node("fallback", fallback_process)

robust_wf.add_edge(START, "process")
robust_wf.add_conditional_edges("process", check_error, {"fallback": "fallback", "done": END})
robust_wf.add_edge("fallback", END)

robust_app = robust_wf.compile()

# Test 1: flaky node recovers on retry 3
flaky_counter["n"] = 0
out = robust_app.invoke({"input_text": "Hello world", "result": "", "error": "", "attempts": 0})
print(f"\nResult: {out['result']}")
print(f"Attempts: {out.get('attempts', 'N/A')}")
print(f"Error: {out.get('error', 'none')}")

In [ ]:
# ── Test 2: all retries fail -> fallback kicks in ─────────────────────────────

@retry_node(max_retries=2, delay=0.05)
def always_fails(state: RobustState) -> dict:
    raise RuntimeError("Permanent failure")


fail_wf = StateGraph(RobustState)
fail_wf.add_node("process", always_fails)
fail_wf.add_node("fallback", fallback_process)
fail_wf.add_edge(START, "process")
fail_wf.add_conditional_edges("process", check_error, {"fallback": "fallback", "done": END})
fail_wf.add_edge("fallback", END)

fail_app = fail_wf.compile()
out2 = fail_app.invoke({"input_text": "test", "result": "", "error": "", "attempts": 0})
print(f"Result: {out2['result']}")
print("All retries exhausted, fallback handled it gracefully.")

In [ ]:
# ── Dead-letter pattern ───────────────────────────────────────────────────────
# Record failures and continue processing other items.

class BatchState(TypedDict):
    items: list[str]
    results: list[str]
    dead_letter: list[str]


def process_batch(state: BatchState) -> dict:
    results = []
    dead_letter = []
    for item in state["items"]:
        try:
            # Simulate: items containing "bad" will fail
            if "bad" in item.lower():
                raise ValueError(f"Cannot process '{item}'")
            results.append(f"OK: {item}")
        except Exception as e:
            dead_letter.append(f"FAILED: {item} ({e})")
    return {"results": results, "dead_letter": dead_letter}


batch_wf = StateGraph(BatchState)
batch_wf.add_node("process", process_batch)
batch_wf.add_edge(START, "process")
batch_wf.add_edge("process", END)

batch_app = batch_wf.compile()
out3 = batch_app.invoke({
    "items": ["good item 1", "bad item", "good item 2", "bad item 2"],
    "results": [],
    "dead_letter": [],
})

print("Results:")
for r in out3["results"]:
    print(f"  {r}")
print("\nDead letter queue:")
for d in out3["dead_letter"]:
    print(f"  {d}")

---
## Summary

| Pattern | What it solves |
|---------|---------------|
| **State machine** | Explicit states, resumability |
| **Graph-based workflow** | Branching, conditional routing |
| **LangGraph nodes/edges** | Framework support for state, streaming, visualization |
| **Conditional routing** | Content-aware path selection |
| **Interrupts / HITL** | Human approval gates |
| **Checkpointing** | Resume, replay, fork executions |
| **Retry + fallback** | Resilience against transient failures |
| **Dead letter** | Graceful degradation for batch processing |

---
## 10. Exercises

### Exercise 1 — Conceptual

Answer the following in your own words (add markdown cells below each question):

1. **Why does ad-hoc chaining fail for production agent systems?** Name at least three failure modes.
2. **What is the difference between a state machine and a graph-based workflow?** When would you choose one over the other?
3. **Explain the role of checkpointing in a HITL workflow.** What happens if you skip checkpointing and the server restarts during a human review?

*Your answers here...*

### Exercise 2 — Coding: Build a Multi-Step Research Workflow

Build a LangGraph workflow that processes a research question through these steps:

1. **Plan** — break the question into 2-3 sub-questions  
2. **Research** — "look up" each sub-question (use mock_llm)  
3. **Synthesize** — combine the findings into a coherent answer  
4. **Quality Check** — if the answer is too short (< 50 chars), loop back to Research; otherwise finish  

Requirements:
- Use `StateGraph` with a proper `TypedDict` state
- Include at least one **conditional edge** (the quality check loop)
- Add a `MemorySaver` checkpointer
- Print the execution trace

In [ ]:
# ── Exercise 2: your code here ────────────────────────────────────────────────

class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    findings: list[str]
    answer: str
    iteration: int


# TODO: implement plan_node, research_node, synthesize_node, quality_check
# TODO: build the graph with conditional looping
# TODO: run it and print the trace

### Exercise 3 — Design: Orchestration System

You are building a **document processing pipeline** for a law firm. Documents arrive as PDFs and must go through:

1. **OCR / Text extraction**  
2. **Classification** (contract, brief, correspondence, other)  
3. **Entity extraction** (parties, dates, amounts)  
4. **Risk flagging** (if contract, check for unusual clauses)  
5. **Human review** (if risk flags found)  
6. **Storage** (index in vector DB)  

Design this system on paper (or in markdown cells below):

- Draw the graph (nodes and edges, including conditional edges)
- Identify which nodes need **HITL interrupts**
- Identify which nodes need **retry/fallback** (hint: OCR and LLM calls)
- Where would you place **checkpoints** for cost efficiency?
- How would you handle a batch of 1,000 documents? (hint: dead-letter queue)

*Your design here...*

---

**End of Chapter 9 Lab.** Next up: Chapter 10 — Memory.